# Fine-tune the AML Alert Investigator on IBM data (QLoRA)
Same base model (`Qwen/Qwen2.5-1.5B-Instruct`) and QLoRA setup as `finetune_qlora.ipynb`, but trained on the IBM typology examples (`data/ibm_sft`).
Runtime > Change runtime type > **T4 GPU**. Run the cells top to bottom. Training takes roughly 1 to 1.5 hours; scoring the base and fine-tuned models takes about 20 minutes more.

**If Colab disconnects:** reconnect, run sections 1 to 3 again (Drive, code, data), then re-run the *same* training cell. It resumes from the newest checkpoint on Drive (saved every 25 steps).

**Before you start:** on your laptop run `python -m src.ibm.build_sft`, then upload the three files in `data/ibm_sft/` to Google Drive at `MyDrive/aml-ibm-finetune/ibm_sft/`.

In [ ]:
!nvidia-smi | head -n 12

## 1. Connect Google Drive
Checkpoints, the adapter, the data and the results all live on Drive, so nothing is lost when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = '/content/drive/MyDrive/aml-ibm-finetune'
RUN_FOLDER = DRIVE_FOLDER + '/run1'            # checkpoints and the final adapter
RESULTS_FOLDER = DRIVE_FOLDER + '/results'     # model answers on the test split
!mkdir -p {RUN_FOLDER} {RESULTS_FOLDER}

## 2. Get the code
If the repo is private, open the key icon (Secrets), add `GITHUB_TOKEN` (a fine-grained, read-only token for this repo) and switch on notebook access. If it is public, delete the two token lines and clone `https://github.com/coachuttam/aml-ibm-finetune.git` directly.
Push your latest commits first: Colab clones from GitHub.

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/coachuttam/aml-ibm-finetune.git
%cd aml-ibm-finetune
!pip install -q -r requirements-train.txt

## 3. Load the data from Drive
Copies `train.jsonl`, `val.jsonl` and `test.jsonl` from Drive into the repo and checks the counts. Expect about 1391 / 284 / 656.

In [ ]:
!mkdir -p data/ibm_sft
!cp {DRIVE_FOLDER}/ibm_sft/*.jsonl data/ibm_sft/
!wc -l data/ibm_sft/*.jsonl

## 4. Smoke test (about 3 minutes)
Trains on 32 examples to prove everything runs, including the length check, before the long run. It writes to `/content/smoke`, not to Drive.

In [ ]:
!python -m src.ibm.finetune_ibm --output-dir /content/smoke --limit 32 --epochs 1

## 5. Full training (resumable)
Watch `loss` fall and `eval_loss` follow it. If `eval_loss` rises while `loss` keeps falling, stop: the model is memorising.
Checkpoints go to `run1` on Drive every 25 steps. **After a disconnect, run this same cell again** and it continues from the last checkpoint.

In [ ]:
!python -m src.ibm.finetune_ibm --output-dir {RUN_FOLDER}

## 6. Score the base model and the fine-tuned model on the test split
Each run writes the model's answers to Drive. They are scored on your laptop (no GPU needed). The fine-tuned run reads the adapter saved in step 5.

In [ ]:
!python -m src.ibm.eval_sft generate --output {RESULTS_FOLDER}/ibm_sft_base_test.jsonl

In [ ]:
!python -m src.ibm.eval_sft generate --adapter {RUN_FOLDER}/adapter --output {RESULTS_FOLDER}/ibm_sft_finetuned_test.jsonl

## 7. Result
Download the two `ibm_sft_*_test.jsonl` files from `MyDrive/aml-ibm-finetune/results/` into your laptop's `results/` folder, then run `python -m src.ibm.eval_sft score`.

In [ ]:
!ls -lh {RUN_FOLDER}/adapter {RESULTS_FOLDER}